# Interactive prediction

## Imports and dependencies

In [ ]:
from pathlib import Path
import sys

import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from hordesofhell.models.simple_dataloader import create_simple_dataloaders
from hordesofhell.models.simple_nn import SimpleNN

CSV_PATH = PROJECT_ROOT / "simple_output.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "best_model.pth"

## Load model and necessary components

In [ ]:
_, _, meta = create_simple_dataloaders(
    csv_path=CSV_PATH,
    batch_size=16,
    train_split=0.8,
    seed=42,
)

difficulty_to_index = meta["difficulty_to_index"]
choice_to_index = meta["choice_to_index"]

model = SimpleNN(input_features=meta["input_features"])
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}. Train and save model weights first."
    )

state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded model from: {CHECKPOINT_PATH}")
print(f"Allowed difficulty values: {difficulty_to_index.keys()}")
print(f"Allowed choice values: {choice_to_index.keys()}")

In [ ]:
def build_feature_tensor(
    difficulty: str, run_id: int, round_id: int, choice: str
) -> torch.Tensor:
    difficulty_clean = difficulty.strip()
    choice_clean = choice.strip().lower()

    if difficulty_clean not in difficulty_to_index:
        raise ValueError(
            f"Unknown difficulty '{difficulty}'. Valid: {sorted(difficulty_to_index.keys())}"
        )
    if choice_clean not in choice_to_index:
        raise ValueError(
            f"Unknown choice '{choice}'. Valid: {sorted(choice_to_index.keys())}"
        )

    difficulty_vec = torch.zeros(len(difficulty_to_index), dtype=torch.float32)
    difficulty_vec[difficulty_to_index[difficulty_clean]] = 1.0

    choice_vec = torch.zeros(len(choice_to_index), dtype=torch.float32)
    choice_vec[choice_to_index[choice_clean]] = 1.0

    numeric_vec = torch.tensor(
        [float(run_id), float(round_id)], dtype=torch.float32
    )
    features = torch.cat((difficulty_vec, numeric_vec, choice_vec), dim=0)
    return features.unsqueeze(0)


def predict_total_aether(
    difficulty: str, run_id: int, round_id: int, choice: str
) -> float:
    x = build_feature_tensor(difficulty, run_id, round_id, choice)
    with torch.no_grad():
        pred = model(x)
    return float(pred.item())

## Interactive prediction

In [ ]:
difficulty_in = "T4"  # T1-T4
run_id_in = 21  # 0>
round_id_in = 2  # 0-10
choice_in = "The Chaotic Offering"

predicted_total_aether = predict_total_aether(
    difficulty=difficulty_in,
    run_id=run_id_in,
    round_id=round_id_in,
    choice=choice_in,
)

print(f"Predicted Total Aether: {predicted_total_aether:.2f}")

All offerings with the same information

In [ ]:
print("Predictions:")
print("Offering | Predicted Total Aether")
print("-------------------------------")
for choice in choice_to_index.keys():
    predicted_total_aether = predict_total_aether(
        difficulty=difficulty_in,
        run_id=run_id_in,
        round_id=round_id_in,
        choice=choice,
    )
    print(f"{choice} | {predicted_total_aether:.2f}")